In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import polars as pl
from polars import col, lit, when
import re

import sys
import os

root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from scripts.feature_calculation import build_processed_dataset
from scripts.feature_engineering import compute_global_stats
from scripts.train_eval_model import train_baseline

### Creating train dataframe with new features

При считывании создаем колонку, по которой можно отличить train от pretrain и приводим данные к одинаковой схеме

In [3]:
train_1 = pl.scan_parquet('../../data/train_part_1.parquet')
train_1 = train_1.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_1 = pl.scan_parquet('../../data/pretrain_part_1.parquet')
pretrain_1 = pretrain_1.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_2 = pl.scan_parquet('../../data/train_part_2.parquet')
train_2 = train_2.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_2 = pl.scan_parquet('../../data/pretrain_part_2.parquet')
pretrain_2 = pretrain_2.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_3 = pl.scan_parquet('../../data/train_part_3.parquet')
train_3 = train_3.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_3 = pl.scan_parquet('../../data/pretrain_part_3.parquet')
pretrain_3 = pretrain_3.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

pretest = pl.scan_parquet('../../data/pretest.parquet')
pretest = pretest.with_columns(
    pl.lit(0).alias('is_train')
)
test = pl.scan_parquet('../../data/test.parquet')
test = test.with_columns(
    pl.lit(1).alias('is_train')
)

labels = pl.scan_parquet('../../data/train_labels.parquet')

In [4]:
full_train = pl.concat([pretrain_1, pretrain_2, pretrain_3, 
                        train_1, train_2, train_3, pretest, test], how='vertical') 

In [5]:
import shutil

# Compute population-level statistics from the full pre-test history
# (pretrain + train). This covers all data available before the test period,
# giving the most stable estimates for global frequencies and amount
# distributions. The resulting tables are frozen here and injected into
# build_processed_dataset so that Sections F and G use proper training-set
# statistics instead of falling back to per-customer cumulative proxies.
#
# Note: pretest and test are intentionally excluded — they contain transactions
# from the test period and must not influence the population-level statistics
# used to score those same transactions.
history_lf = pl.concat([pretrain_1, pretrain_2, pretrain_3,
                         train_1,    train_2,    train_3])
global_stats = compute_global_stats(history_lf)

# Clear the existing output so that partitions built without global_stats
# do not persist alongside the newly generated ones.
shutil.rmtree('../data_processed/', ignore_errors=True)
shutil.rmtree('../data_splits/', ignore_errors=True)

build_processed_dataset(full_train, global_stats=global_stats)

  100,000 customers → 50 partitions × ~2,000 customers each
[██████████████████████████████] 100.0%  part 50/50  (1,707,096 rows)  elapsed 1m07s  ETA 0s              

Done. 86,311,523 total rows written across 50 files in '../data_processed/'.


Посмотрим что получилось

In [6]:
example_data = pl.read_parquet('../data_processed/part_0000.parquet')
example_data.head()

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,is_train,hour,day_of_week,day_of_month,week_of_year,hour_of_day,is_weekend,is_night,is_working_hour,minutes_from_midnight,log_amount,amount_abs,amount_round_100,amount_round_1000,…,velocity_change_flag,time_gap_variance_30d,time_gap_mean_30d,time_gap_min_10tx,new_device_flag,new_mcc_flag,new_channel_flag,merchant_entropy_user,time_gap_cv_30d,browser_language_mismatch,new_device_and_night_flag,rdp_and_large_amount_flag,amount_zscore_given_channel,amount_zscore_given_mcc,amount_zscore_given_device,tx_time_zscore_given_user,amount_zscore_channel,amount_zscore_mcc,global_combination_freq,session_first_tx_flag,language_change_flag,os_change_flag,timezone_change_flag,rapid_sequence_flag,suspicious_env_flag,mcc_rare_global_flag,rare_combination_flag,device_change_and_large_amount_flag,voip_and_new_mcc_flag,compromised_and_high_amount_flag,session_first_tx_large_flag,geo_jump_proxy,device_entropy_ratio,language_mismatch,timezone_mismatch,merchant_last_seen_days,mcc_last_seen_days
i64,i64,datetime[μs],i32,i32,i32,i32,f32,i32,str,i32,str,str,i32,i64,i32,str,str,str,str,i32,i32,str,i32,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,f32,i8,i8,…,i8,f32,f32,i32,i8,i8,i8,f32,f32,i8,i8,i8,f32,f32,f32,f32,f32,f32,u32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i32,f32,i8,i8,i32,i32
123123123124290,123939168233221,2024-10-01 00:03:20,7,56,3,4,null,null,null,null,"""ru""",null,3,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,3,0.0,0.0,1,1,…,0,1548.186523,959.555542,0,0,0,0,6.961049e6,1.613441,0,0,0,1.409423e6,3.8061e6,3.270069e6,4.373639,-0.017505,null,0,0,0,0,0,0,0,null,1,0,0,0,0,0,0.027027,1,0,0,0
123140302994545,123861859323955,2024-10-01 00:03:20,7,56,4,15,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,3,0.0,0.0,1,1,…,0,684.982666,417.955566,1,0,0,0,5023353.5,1.638889,0,0,0,408391.25,375927.78125,342694.21875,3.876753,-0.013278,null,0,0,0,0,0,0,0,null,1,0,0,0,0,0,0.026316,0,1,0,0
123131713060513,126533330625689,2024-10-01 00:05:46,14,75,6,5,23267.0,0,"""4""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,5,10.054834,23267.0,0,0,…,0,758.181946,428.433319,0,0,0,0,0.111801,1.769661,0,0,0,60229.402344,59146.171875,86628.0,7.588052,-0.095625,-0.045366,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.025641,0,1,0,1
123123123126335,125107399907706,2024-10-01 00:07:13,14,75,6,5,22498.0,0,null,3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,7,10.021226,22498.0,0,0,…,0,379.917603,194.633331,0,0,0,0,2542905.5,1.951966,0,0,0,158851.875,244043.8125,194970.734375,5.509225,-0.095855,null,0,0,0,0,0,0,0,null,1,0,0,0,0,0,0.025,0,1,0,0
123131713060293,126181143683685,2024-10-01 00:09:25,7,56,3,4,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,9,0.0,0.0,1,1,…,0,754.676819,359.244446,0,0,0,0,2692769.5,2.100733,0,0,0,273316.1875,233719.609375,204013.21875,4.505178,-0.017505,null,0,0,0,0,0,0,0,null,1,0,0,0,0,0,0.02439,0,1,0,0


### Training baseline on engineered features

In [7]:
train_baseline()

Found 50 parquet partitions in '../data_processed'

  Cache miss — missing: ['memmap_meta.json', 'X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy', 'is_labeled_val.npy']
Loading labels …
  87,514 labelled rows  |  51,438 positives  (58.777%)

Pre-scan: counting train/val rows for memmap allocation …
  [███████████████████████████████████] 100.0%  50/50  ETA 0s  train 60,916,817 | val 24,761,023          
  train rows: 60,916,817  |  val rows: 24,761,023

Detecting feature count from first chunk …
  260 feature columns detected.

Allocating memmap files …
  X_train.npy : 60,916,817 × 260  ≈ 63.4 GB on disk
  X_val.npy   : 24,761,023 × 260  ≈ 25.8 GB on disk

Writing chunks into memmap files …
  [███████████████████████████████████] 100.0%  50/50  ETA 0s  written train 60,916,817 | val 24,761,023             

  Train : 60,916,817 rows  | 37,752 positives (0.0620%)
  Val   : 24,761,023 rows  | 13,686 positives (0.0553%)

  Metadata saved → ..\data_splits\memmap_meta.json
  Feature c

In [ ]:
# Final PR-AUC    : 0.001098

In [8]:
sub = pd.read_csv('submission.csv')
sub.shape

(633683, 2)

In [9]:
sub.head()

,event_id,predict
0,125390866897300,0.004591
1,126189731373139,0.113199
2,125081630705881,0.002135
3,125262020316273,0.000580
4,125682927274458,0.008564


### Feature importance analysis

In [11]:
import lightgbm as lgb

bst = lgb.Booster(model_file='baseline_lgbm.txt')

In [22]:
model = lgb.Booster(model_file='baseline_lgbm.txt')

In [23]:
model.feature_name()

['event_type_nm',
 'event_desc',
 'channel_indicator_type',
 'channel_indicator_sub_type',
 'operaton_amt',
 'currency_iso_cd',
 'pos_cd',
 'timezone',
 'session_id',
 'operating_system_type',
 'phone_voip_call_state',
 'web_rdp_connection',
 'hour',
 'day_of_week',
 'day_of_month',
 'week_of_year',
 'hour_of_day',
 'is_weekend',
 'is_night',
 'is_working_hour',
 'minutes_from_midnight',
 'log_amount',
 'amount_abs',
 'amount_round_100',
 'amount_round_1000',
 'amount_is_integer',
 'amount_missing_flag',
 'amount_currency_mismatch_flag',
 'amount_usd_normalized',
 'is_month_start',
 'is_month_end',
 'is_payday',
 'is_holiday',
 'tx_count_lifetime',
 'time_since_last_tx_minutes',
 'mcc_freq_user_cum',
 'pos_freq_user_cum',
 'event_desc_user_freq',
 'event_type_user_freq',
 'mcc_is_new_for_user',
 'pos_cd_is_new',
 'mcc_transaction_share_user',
 'pos_cd_transaction_share_user',
 'merchant_switch_flag',
 'time_since_last_3_tx_mean',
 'mcc_frequency_user',
 'event_desc_is_new_for_user',
 '

In [24]:
importances = importances = model.feature_importance(importance_type='gain')
feature_importance_df = pd.DataFrame({
    'Feature': model.feature_name(),
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [25]:
feature_importance_df.head(30)

,Feature,Importance
44,time_since_last_3_tx_mean,137966.299146
213,global_mcc_freq,90486.215285
37,event_desc_user_freq,90138.605416
190,spend_in_channel_lifetime,61330.408890
220,global_event_desc_freq,59363.082040
4,operaton_amt,58881.362158
237,amount_zscore_given_device,58240.613001
225,time_gap_mean_30d,54442.208632
48,event_desc_share_user,49832.267904
239,amount_zscore_channel,45464.942554


In [26]:
feature_importance_df.to_csv('feature_importances.csv', index=False)